# Phase 5 – Suivi des candidatures (TrackerAgent)

Ce notebook implémente le **TrackerAgent** du système RecruitAI.
Son rôle est d'assurer le suivi des candidatures, en conservant
l'historique des actions, des documents générés et des statuts associés.

Cette étape permet de transformer RecruitAI en un véritable assistant
de recrutement de bout en bout.


In [1]:
import json
import os
from datetime import datetime
from typing import Dict, Any, List
import pandas as pd


In [2]:
TRACKING_PATH = "data/tracking/applications.json"
os.makedirs(os.path.dirname(TRACKING_PATH), exist_ok=True)

def load_applications():
    if not os.path.exists(TRACKING_PATH):
        return []
    with open(TRACKING_PATH, "r", encoding="utf-8") as f:
        return json.load(f)

def save_applications(apps):
    with open(TRACKING_PATH, "w", encoding="utf-8") as f:
        json.dump(apps, f, indent=2, ensure_ascii=False)


In [3]:
class TrackerAgent:
    """
    Agent responsable du suivi des candidatures
    """

    def __init__(self):
        self.applications = load_applications()

    def add_application(
        self,
        offer: Dict[str, Any],
        matching_score: float,
        cv_path: str,
        letter_path: str
    ):
        application = {
            "application_id": len(self.applications) + 1,
            "offer_title": offer.get("offer_title"),
            "company": offer.get("company"),
            "matching_score": matching_score,
            "status": "Applied",
            "created_at": datetime.now().isoformat(),
            "last_update": datetime.now().isoformat(),
            "documents": {
                "cv": cv_path,
                "cover_letter": letter_path
            },
            "history": [
                {
                    "date": datetime.now().isoformat(),
                    "event": "Application created"
                }
            ]
        }

        self.applications.append(application)
        save_applications(self.applications)
        return application

    def update_status(self, application_id: int, new_status: str):
        for app in self.applications:
            if app["application_id"] == application_id:
                app["status"] = new_status
                app["last_update"] = datetime.now().isoformat()
                app["history"].append({
                    "date": datetime.now().isoformat(),
                    "event": f"Status changed to {new_status}"
                })
                save_applications(self.applications)
                return app
        raise ValueError("Application not found")

    def list_applications(self):
        return self.applications


In [4]:
tracker = TrackerAgent()

# Exemple de donnée venant du MatcherAgent
best_match_example = {
    "offer_title": "Data Scientist Biology",
    "company": "Company X"
}

application = tracker.add_application(
    offer=best_match_example,
    matching_score=82.5,
    cv_path="cv_generated.pdf",
    letter_path="cover_letter_generated.pdf"
)

application


{'application_id': 1,
 'offer_title': 'Data Scientist Biology',
 'company': 'Company X',
 'matching_score': 82.5,
 'status': 'Applied',
 'created_at': '2025-12-13T18:15:10.104793',
 'last_update': '2025-12-13T18:15:10.104802',
 'documents': {'cv': 'cv_generated.pdf',
  'cover_letter': 'cover_letter_generated.pdf'},
 'history': [{'date': '2025-12-13T18:15:10.104805',
   'event': 'Application created'}]}

In [5]:
tracker.update_status(
    application_id=1,
    new_status="Interview Scheduled"
)


{'application_id': 1,
 'offer_title': 'Data Scientist Biology',
 'company': 'Company X',
 'matching_score': 82.5,
 'status': 'Interview Scheduled',
 'created_at': '2025-12-13T18:15:10.104793',
 'last_update': '2025-12-13T18:15:16.894788',
 'documents': {'cv': 'cv_generated.pdf',
  'cover_letter': 'cover_letter_generated.pdf'},
 'history': [{'date': '2025-12-13T18:15:10.104805',
   'event': 'Application created'},
  {'date': '2025-12-13T18:15:16.894795',
   'event': 'Status changed to Interview Scheduled'}]}

In [6]:
apps = tracker.list_applications()
df_apps = pd.DataFrame(apps)

df_apps[[
    "application_id",
    "offer_title",
    "company",
    "matching_score",
    "status",
    "created_at"
]]


,application_id,offer_title,company,matching_score,status,created_at
0,1,Data Scientist Biology,Company X,82.5,Interview Scheduled,2025-12-13T18:15:10.104793


In [7]:
df_apps["status"].value_counts()


,count
status,
Interview Scheduled,1
